# 02 · Delinquency dynamics and vintage cohorts
**Default definition (credit level):** first month at 90+ DPD (bureau `STATUS` 4 or 5).

**Questions:** how do credits move between delinquency buckets month to month, where does the portfolio roll forward fastest, and which cohorts deteriorate faster?

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import duckdb
import pandas as pd
GOLD, SILVER, REPORTS = ROOT / "data/gold", ROOT / "data/silver", ROOT / "reports/data_studio/data"
def q(sql: str) -> pd.DataFrame:
    return duckdb.sql(sql.replace("{gold}", GOLD.as_posix()).replace("{silver}", SILVER.as_posix())).df()

## Transition matrix (all credit types)

In [ ]:
order = ["Current", "DPD 1-30", "DPD 31-60", "DPD 61-90", "DPD 91-120", "DPD 120+"]
t = pd.read_csv(REPORTS / "transition_matrix.csv")
m = (t[t.credit_type == "All"].pivot_table(index="from_state", columns="to_state", values="probability")
       .reindex(index=order, columns=order + ["Closed"]).fillna(0))
m.style.format("{:.1%}").background_gradient(cmap="Blues", axis=None)

## Roll rates by credit type

In [ ]:
roll = pd.read_csv(REPORTS / "roll_rates.csv")
roll.pivot_table(index="from_state", columns="credit_type", values="pct_roll_forward").reindex(order[:-1]).style.format("{:.1%}")

## Vintage curves
Kaplan–Meier cumulative 90+ DPD rate. Points with fewer than 100 credits at risk are excluded.

In [ ]:
from IPython.display import Image
Image(filename=str(ROOT / "images/vintage_curves.png"))

In [ ]:
v = pd.read_csv(REPORTS / "vintage_curves.csv")
v = v[(v.credit_type == "All") & (v.is_reliable == 1) & (v.months_on_book.isin([6, 12, 24, 36]))]
v.pivot_table(index="vintage_cohort", columns="months_on_book", values="cumulative_default_rate").style.format("{:.2%}")

## Notes
_Which bucket has the highest roll-forward rate? What is the cure rate from DPD 1-30? Which cohort and credit type deteriorate fastest? Write the answers here after running._